# Download Llama Base + LoRA to Local Paths

This notebook downloads:
- Base model: `meta-llama/Llama-3.1-8B-Instruct`
- LoRA adapter: `NingLab/GeLLMO-C-P10-Llama`

Target paths:
- `models/Llama/base`
- `models/Llama/gellmoC-lora`

In [1]:
import os
from pathlib import Path

from huggingface_hub import login, snapshot_download

BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
LORA_MODEL_ID = "NingLab/GeLLMO-C-P10-Llama"

BASE_LOCAL_DIR = Path("models/Llama/base")
LORA_LOCAL_DIR = Path("models/Llama/gellmoC-lora")

BASE_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
LORA_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

print("Base path:", BASE_LOCAL_DIR.resolve())
print("LoRA path:", LORA_LOCAL_DIR.resolve())

Base path: /work/nvme/bfuy/rgao7/Molo/models/Llama/base
LoRA path: /work/nvme/bfuy/rgao7/Molo/models/Llama/gellmoC-lora


/u/rgao7/.conda/envs/molo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Set HF_TOKEN in your shell before opening Jupyter:
HF_TOKEN = os.environ.get("HF_TOKEN", "")
hf_token = HF_TOKEN
if not hf_token:
    raise ValueError("HF_TOKEN is empty. Please export it in your shell before running this notebook.")

login(token=hf_token, add_to_git_credential=False)
print("Hugging Face login OK")


Hugging Face login OK


In [5]:
print("Downloading base model:", BASE_MODEL_ID)
snapshot_download(
    repo_id=BASE_MODEL_ID,
    local_dir=str(BASE_LOCAL_DIR),
    token=hf_token,
    local_dir_use_symlinks=False,
    resume_download=True,
)
print("Base model downloaded to", BASE_LOCAL_DIR.resolve())

/u/rgao7/.conda/envs/molo/lib/python3.11/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
/u/rgao7/.conda/envs/molo/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 17 files: 100%|██████████| 17/17 [23:04<00:00, 81.46s/it] 

Base model downloaded to /work/nvme/bfuy/rgao7/Molo/models/Llama/base


In [6]:
print("Downloading LoRA adapter:", LORA_MODEL_ID)
snapshot_download(
    repo_id=LORA_MODEL_ID,
    local_dir=str(LORA_LOCAL_DIR),
    token=hf_token,
)
print("LoRA adapter downloaded to", LORA_LOCAL_DIR.resolve())

Fetching 5 files: 100%|██████████| 5/5 [00:02<00:00,  1.77it/s]

LoRA adapter downloaded to /work/nvme/bfuy/rgao7/Molo/models/Llama/gellmoC-lora


In [7]:
base_files = sorted([p.name for p in BASE_LOCAL_DIR.glob("*")])
lora_files = sorted([p.name for p in LORA_LOCAL_DIR.glob("*")])

print(f"Base files: {len(base_files)}")
print(base_files[:20])
print()
print(f"LoRA files: {len(lora_files)}")
print(lora_files[:20])

Base files: 16
['.cache', '.gitattributes', 'LICENSE', 'README.md', 'USE_POLICY.md', 'config.json', 'generation_config.json', 'model-00001-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'model.safetensors.index.json', 'original', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']

LoRA files: 6
['.cache', '.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'config.json']


In [1]:

import torch
from pathlib import Path
import sys

repo_root = Path.cwd()         
sys.path.append(str(repo_root.parent))

from Molo.utils.load_model import load_model_tokenizer
from Molo.rl.ppo.actor import MoloActor

device = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer = load_model_tokenizer(
    base_model="Llama",
    base_model_path="models/Llama/base",
    lora_adapter_path="models/Llama/gellmoC-lora",
    use_lora=True,
    device=device,
)
actor = MoloActor(model, tokenizer, device=device)

print("model loaded on:", device)


/u/rgao7/.conda/envs/molo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards:   0%|          | 0/4 [00:38<?, ?it/s]


KeyboardInterrupt: 